# Romania 2001: national originating-school applicant pages

These are the **Gymnasium Applicant View** pages reached through the Ministry’s school directory. Each page gives an applicant’s name, admission score, examination score, and grades 5–8 average, but no destination or admitted/rejected result. The Ministry menu also has a **different** “admitted + rejected per school” link leading to school-result pages with destination/program columns. Use the companion school-results-and-incoming notebook for those distinct reports. This notebook only collects and verifies the score-bearing applicant webpages.

Bucharest’s **289/289** pages and the earlier CS, GL, TL, AR, and SB pages are already saved. Alba’s **168** pages live in its earlier verified cache. This notebook targets the other 40 counties and reuses every saved standard page. A page that fails remains unresolved for later retry; no student is labeled rejected merely because a page is missing.

**Run cells in order.** Cell 1 has your settings; Cell 2 previews the work without network access; Cell 3 starts or resumes acquisition when you run it; Cell 4 shows the saved progress. Use the `sports_net` kernel and keep other Wayback download notebooks stopped while Cell 3 runs. Pages are saved immediately to `~/Desktop/VECTOR_temp`, outside Dropbox and Git. The repository receives only name-free progress counts.


In [ ]:
# CELL 1 — SETTINGS YOU CONTROL. This cell makes no web request.
from pathlib import Path
import sys, importlib

REPO_ROOT = Path("/Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE")
ANALYSIS_ROOT = REPO_ROOT / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
CODE_DIR = ANALYSIS_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
worker = importlib.reload(importlib.import_module("EDUCATION_20261002_romania_three_county_gymnasium_acquisition"))

# All 40 counties not named AB. Alba's complete pages are already in a separate cache.
# For a smaller run, replace with a tuple such as ("B", "AG", "BV").
COUNTIES = tuple(worker.DEFAULT_COUNTIES)
RUN_DOWNLOAD = True  # Only Cell 3 contacts Wayback, and only when you run it.

# Each run stops at the first reached cap. Run Cell 3 again to resume.
MAX_NEW_PAGES = 800
MAX_HTTP_ATTEMPTS = 1200
MAX_HOURS = 12.0

# Retain the archive pacing and server-error policy used in the successful pilots.
MIN_INTERVAL_SECONDS = 22.0
MAX_INTERVAL_SECONDS = 50.0
FIRST_429_COOLDOWN_SECONDS = 600
SECOND_429_COOLDOWN_SECONDS = 1500
SERVER_ERROR_WAIT_SECONDS = 10.0

STATUS_FILE = ANALYSIS_ROOT / "outputs/romania_2001_national_gymnasium_expansion_20261006/gymnasium_acquisition_status.csv"
assert set(COUNTIES).issubset(set(worker.COUNTIES))
print(f"Counties selected: {len(COUNTIES)} — {', '.join(COUNTIES)}")
print("Download enabled:", RUN_DOWNLOAD)
print("Private page folder:", worker.PAGES)
print("Name-free status file:", STATUS_FILE)
print("No web request has been made by this cell.")

## Cell 2 — local preview

Read the saved Ministry school directories and verify pages already on disk. This can take a few seconds for thousands of pages. It makes **no** web request and writes no file. The total includes school pages that previously failed; those stay open for another pass.

In [ ]:
# CELL 2 — READ-ONLY INVENTORY; NO NETWORK REQUEST
preview_args = ["--counties", *COUNTIES, "--status-file", str(STATUS_FILE)]
worker.main(preview_args)

## Cell 3 — researcher-controlled download

Running this cell is the only action that contacts Wayback. Before each request, the log prints the county, originating-school code, and full URL address. After a successful download, it prints **NEW SOURCE SAVED** and the number of applicant rows. Pauses name their reason. The downloader checkpoints and verifies every page before moving on. A third consecutive HTTP 429 or a run cap stops safely; rerun this same cell later to continue.

In [ ]:
# CELL 3 — START OR RESUME SOURCE ACQUISITION
if not RUN_DOWNLOAD:
    print("Download is OFF. Change RUN_DOWNLOAD in Cell 1 and rerun Cell 1.")
else:
    args = [
        "--run", "--counties", *COUNTIES,
        "--status-file", str(STATUS_FILE),
        "--max-new-pages", str(MAX_NEW_PAGES),
        "--max-http-attempts", str(MAX_HTTP_ATTEMPTS),
        "--max-hours", str(MAX_HOURS),
        "--min-interval-seconds", str(MIN_INTERVAL_SECONDS),
        "--max-interval-seconds", str(MAX_INTERVAL_SECONDS),
        "--first-429-cooldown-seconds", str(FIRST_429_COOLDOWN_SECONDS),
        "--second-429-cooldown-seconds", str(SECOND_429_COOLDOWN_SECONDS),
        "--server-error-wait-seconds", str(SERVER_ERROR_WAIT_SECONDS),
    ]
    print("Starting source-only school-page acquisition for", len(COUNTIES), "counties", flush=True)
    print("Stop limits:", MAX_NEW_PAGES, "new pages;", MAX_HTTP_ATTEMPTS,
          "HTTP attempts;", MAX_HOURS, "hours", flush=True)
    worker.main(args)

## Cell 4 — saved progress after a run or interruption

This cell reads only the name-free progress CSV. It counts verified and unresolved school webpages by county. A school webpage with zero applicant rows still counts as verified if its source identity and hash passed. An unresolved URL address is never interpreted as an empty school.

In [ ]:
# CELL 4 — NAME-FREE PROGRESS; NO WEB REQUEST
import pandas as pd
if not STATUS_FILE.exists():
    print("No progress file yet. Run Cell 3 to create one, or use Cell 2 for a read-only preview.")
else:
    progress = pd.read_csv(STATUS_FILE, dtype={"county": str, "gymnasium_code": str}).fillna("")
    progress = progress.loc[progress["county"].isin(COUNTIES)].copy()
    table = (progress.assign(verified=progress["status"].eq("saved_verified"),
                             unresolved=progress["status"].str.startswith("unresolved"))
             .groupby("county", as_index=False)
             .agg(school_pages=("gymnasium_code", "size"), verified=("verified", "sum"),
                  unresolved=("unresolved", "sum")))
    table["remaining"] = table["school_pages"] - table["verified"]
    display(table)
    print("Verified", int(table["verified"].sum()), "of", int(table["school_pages"].sum()),
          "selected school webpages.")
    print("Unresolved URL addresses:", int(table["unresolved"].sum()))
    print("Status file:", STATUS_FILE)